In [1]:
import numpy as np
import pandas as pd

In [2]:
train_df = pd.read_csv(r"C:\Users\ni\Desktop\Demand Forecaster\raw_data\train.csv")
train_df

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.000,0
1,1,2013-01-01,1,BABY CARE,0.000,0
2,2,2013-01-01,1,BEAUTY,0.000,0
3,3,2013-01-01,1,BEVERAGES,0.000,0
4,4,2013-01-01,1,BOOKS,0.000,0
...,...,...,...,...,...,...
3000883,3000883,2017-08-15,9,POULTRY,438.133,0
3000884,3000884,2017-08-15,9,PREPARED FOODS,154.553,1
3000885,3000885,2017-08-15,9,PRODUCE,2419.729,148
3000886,3000886,2017-08-15,9,SCHOOL AND OFFICE SUPPLIES,121.000,8


In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   family       object 
 4   sales        float64
 5   onpromotion  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 137.4+ MB


In [4]:
train_df.describe()

,id,store_nbr,sales,onpromotion
count,3.000888e+06,3.000888e+06,3.000888e+06,3.000888e+06
mean,1.500444e+06,2.750000e+01,3.577757e+02,2.602770e+00
std,8.662819e+05,1.558579e+01,1.101998e+03,1.221888e+01
min,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
25%,7.502218e+05,1.400000e+01,0.000000e+00,0.000000e+00
50%,1.500444e+06,2.750000e+01,1.100000e+01,0.000000e+00
75%,2.250665e+06,4.100000e+01,1.958473e+02,0.000000e+00
max,3.000887e+06,5.400000e+01,1.247170e+05,7.410000e+02


In [5]:
train_df['family'].value_counts()

family
AUTOMOTIVE                    90936
BABY CARE                     90936
BEAUTY                        90936
BEVERAGES                     90936
BOOKS                         90936
BREAD/BAKERY                  90936
CELEBRATION                   90936
CLEANING                      90936
DAIRY                         90936
DELI                          90936
EGGS                          90936
FROZEN FOODS                  90936
GROCERY I                     90936
GROCERY II                    90936
HARDWARE                      90936
HOME AND KITCHEN I            90936
HOME AND KITCHEN II           90936
HOME APPLIANCES               90936
HOME CARE                     90936
LADIESWEAR                    90936
LAWN AND GARDEN               90936
LINGERIE                      90936
LIQUOR,WINE,BEER              90936
MAGAZINES                     90936
MEATS                         90936
PERSONAL CARE                 90936
PET SUPPLIES                  90936
PLAYERS AND ELECTRONI

In [6]:
# No use of ID feature in our model training, so we omit it.

train_df = train_df.drop(columns=["id"])

In [7]:
# Since, our client is a wholesale & retail store, we will need only a selected set of product categories to work with.

selected_families = [
    "GROCERY I",
    "GROCERY II",
    "DAIRY",
    "BEVERAGES",
    "BREAD/BAKERY",
    "PRODUCE",
    "MEATS",
    "POULTRY",
    "SEAFOOD",
    "EGGS",
    "DELI",
    "PREPARED FOODS",
    "FROZEN FOODS",
    "CLEANING",
    "HOME CARE",
    "PERSONAL CARE"
]


In [8]:
train_df = train_df[train_df["family"].isin(selected_families)]

In [9]:
train_df = train_df.reset_index(drop=True)

In [10]:
print(train_df.shape)
print(train_df["family"].unique())

(1454976, 5)
['BEVERAGES' 'BREAD/BAKERY' 'CLEANING' 'DAIRY' 'DELI' 'EGGS'
 'FROZEN FOODS' 'GROCERY I' 'GROCERY II' 'HOME CARE' 'MEATS'
 'PERSONAL CARE' 'POULTRY' 'PREPARED FOODS' 'PRODUCE' 'SEAFOOD']


In [11]:
train_df = train_df.sort_values(by=["store_nbr", "family", "date"])

In [12]:
train_df

,date,store_nbr,family,sales,onpromotion
0,2013-01-01,1,BEVERAGES,0.0,0
864,2013-01-02,1,BEVERAGES,1091.0,0
1728,2013-01-03,1,BEVERAGES,919.0,0
2592,2013-01-04,1,BEVERAGES,953.0,0
3456,2013-01-05,1,BEVERAGES,1160.0,0
...,...,...,...,...,...
1451455,2017-08-11,54,SEAFOOD,0.0,0
1452319,2017-08-12,54,SEAFOOD,1.0,1
1453183,2017-08-13,54,SEAFOOD,2.0,0
1454047,2017-08-14,54,SEAFOOD,0.0,0


In [13]:
"""
Create time-based features from the 'date' column to capture seasonality 
and temporal patterns in retail demand.

Features created:
- day_of_week: Captures weekly patterns (e.g., weekend vs weekday demand)
- month: Captures monthly/seasonal trends
- day_of_month: Helps identify patterns within a month (e.g., salary cycles)
- is_weekend: Binary flag for weekend shopping behavior
"""

# Ensure date is in datetime format
train_df["date"] = pd.to_datetime(train_df["date"])

# Extract time features
train_df["day_of_week"] = train_df["date"].dt.dayofweek
train_df["month"] = train_df["date"].dt.month
train_df["day_of_month"] = train_df["date"].dt.day
train_df["is_weekend"] = train_df["day_of_week"].isin([5, 6]).astype(int)

In [14]:
train_df

,date,store_nbr,family,sales,onpromotion,day_of_week,month,day_of_month,is_weekend
0,2013-01-01,1,BEVERAGES,0.0,0,1,1,1,0
864,2013-01-02,1,BEVERAGES,1091.0,0,2,1,2,0
1728,2013-01-03,1,BEVERAGES,919.0,0,3,1,3,0
2592,2013-01-04,1,BEVERAGES,953.0,0,4,1,4,0
3456,2013-01-05,1,BEVERAGES,1160.0,0,5,1,5,1
...,...,...,...,...,...,...,...,...,...
1451455,2017-08-11,54,SEAFOOD,0.0,0,4,8,11,0
1452319,2017-08-12,54,SEAFOOD,1.0,1,5,8,12,1
1453183,2017-08-13,54,SEAFOOD,2.0,0,6,8,13,1
1454047,2017-08-14,54,SEAFOOD,0.0,0,0,8,14,0


In [15]:
"""
Create lag features to capture historical demand patterns.

Lag features allow the model to learn from past sales:
- lag_1: Previous day's sales (short-term dependency)
- lag_7: Same day last week (weekly seasonality)
- lag_14: Two weeks ago (longer seasonal memory)

Important:
- Features are created per store and product family
- Data must be sorted before applying shifts
"""

# Ensure correct sorting (CRITICAL for lag features)
train_df = train_df.sort_values(by=["store_nbr", "family", "date"])

# Create lag features
train_df["lag_1"] = train_df.groupby(["store_nbr", "family"])["sales"].shift(1)
train_df["lag_7"] = train_df.groupby(["store_nbr", "family"])["sales"].shift(7)
train_df["lag_14"] = train_df.groupby(["store_nbr", "family"])["sales"].shift(14)

In [16]:
train_df

,date,store_nbr,family,sales,onpromotion,day_of_week,month,day_of_month,is_weekend,lag_1,lag_7,lag_14
0,2013-01-01,1,BEVERAGES,0.0,0,1,1,1,0,NaN,NaN,NaN
864,2013-01-02,1,BEVERAGES,1091.0,0,2,1,2,0,0.0,NaN,NaN
1728,2013-01-03,1,BEVERAGES,919.0,0,3,1,3,0,1091.0,NaN,NaN
2592,2013-01-04,1,BEVERAGES,953.0,0,4,1,4,0,919.0,NaN,NaN
3456,2013-01-05,1,BEVERAGES,1160.0,0,5,1,5,1,953.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1451455,2017-08-11,54,SEAFOOD,0.0,0,4,8,11,0,2.0,0.0,4.0
1452319,2017-08-12,54,SEAFOOD,1.0,1,5,8,12,1,0.0,3.0,4.0
1453183,2017-08-13,54,SEAFOOD,2.0,0,6,8,13,1,1.0,0.0,4.0
1454047,2017-08-14,54,SEAFOOD,0.0,0,0,8,14,0,2.0,0.0,4.0


In [17]:
# Drop rows with missing lag values
train_df = train_df.dropna().reset_index(drop=True)

In [18]:
train_df

,date,store_nbr,family,sales,onpromotion,day_of_week,month,day_of_month,is_weekend,lag_1,lag_7,lag_14
0,2013-01-15,1,BEVERAGES,1149.0,0,1,1,15,0,1178.0,1029.0,0.0
1,2013-01-16,1,BEVERAGES,1043.0,0,2,1,16,0,1149.0,1186.0,1091.0
2,2013-01-17,1,BEVERAGES,898.0,0,3,1,17,0,1043.0,847.0,919.0
3,2013-01-18,1,BEVERAGES,1130.0,0,4,1,18,0,898.0,1033.0,953.0
4,2013-01-19,1,BEVERAGES,1280.0,0,5,1,19,1,1130.0,1143.0,1160.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1442875,2017-08-11,54,SEAFOOD,0.0,0,4,8,11,0,2.0,0.0,4.0
1442876,2017-08-12,54,SEAFOOD,1.0,1,5,8,12,1,0.0,3.0,4.0
1442877,2017-08-13,54,SEAFOOD,2.0,0,6,8,13,1,1.0,0.0,4.0
1442878,2017-08-14,54,SEAFOOD,0.0,0,0,8,14,0,2.0,0.0,4.0


In [19]:
"""
Create rolling statistics to capture short-term trends and smooth demand fluctuations.

Rolling features help the model understand:
- recent average demand (trend)
- reduce noise from daily fluctuations

Feature created:
- rolling_mean_7: Average sales over the past 7 days

Important:
- Calculated per store and product family
- Uses past data only (no data leakage)
"""

# Rolling mean of last 7 days (shifted to avoid using current day)
train_df["rolling_mean_7"] = (
    train_df.groupby(["store_nbr", "family"])["sales"]
    .shift(1)
    .rolling(window=7)
    .mean()
)

In [20]:
train_df = train_df.dropna().reset_index(drop=True)

In [21]:
train_df

,date,store_nbr,family,sales,onpromotion,day_of_week,month,day_of_month,is_weekend,lag_1,lag_7,lag_14,rolling_mean_7
0,2013-01-22,1,BEVERAGES,1037.0,0,1,1,22,0,1104.0,1149.0,1029.0,1008.142857
1,2013-01-23,1,BEVERAGES,1052.0,0,2,1,23,0,1037.0,1043.0,1186.0,992.142857
2,2013-01-24,1,BEVERAGES,846.0,0,3,1,24,0,1052.0,898.0,847.0,993.428571
3,2013-01-25,1,BEVERAGES,1103.0,0,4,1,25,0,846.0,1130.0,1033.0,986.000000
4,2013-01-26,1,BEVERAGES,965.0,0,5,1,26,1,1103.0,1280.0,1143.0,982.142857
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1436827,2017-08-11,54,SEAFOOD,0.0,0,4,8,11,0,2.0,0.0,4.0,3.000000
1436828,2017-08-12,54,SEAFOOD,1.0,1,5,8,12,1,0.0,3.0,4.0,3.000000
1436829,2017-08-13,54,SEAFOOD,2.0,0,6,8,13,1,1.0,0.0,4.0,2.714286
1436830,2017-08-14,54,SEAFOOD,0.0,0,0,8,14,0,2.0,0.0,4.0,3.000000


In [22]:
train_df.to_csv(r"C:\Users\ni\Desktop\Demand Forecaster\cleaned_data\cleaned_train.csv", index=False)